# <mark style="display:block; background:#d1c4e9; color:#1a1a1a; padding:6px 12px; border-radius:4px">9/29(화) 오후 · 탐지 룰 ③ · API·HTTP — 실습</mark>

오전에 수상한 IP 하나(`185.220.101.34`)와 두드려 맞은 계정 하나(`admin`)를 찾았습니다.

오후는 그 두 발견에서 출발합니다. **성공을 의심하고 → 밖에 물어보고 → 답이 안 올 때를 읽습니다.**


## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">시작하기</mark>

### 0.1 맨 먼저 · 내 사본 만들기

위 메뉴에서 파일 › 드라이브에 사본 저장을 누릅니다. 제목이 「사본: …」으로 바뀌면 된 것입니다.

### 0.2 오늘 오후의 순서

| 교시 | 무엇 |
|---|---|
| 5교시 | 룰 ③ 심야 성공 접속 · 세 룰을 한 번에 |
| 6교시 | 왜 밖에 묻나 · URL 구조와 메서드 |
| 7교시 | 상태코드 · 401 ↔ 403 · 인증 두 방식 |
| 8교시 | 확인 · 남은 도전 · 내일 예고 |

### 0.3 막혔을 때

1. 문제 아래 **💡 힌트**를 순서대로 따라 합니다.
2. 그래도 막히면 노트북 맨 아래 「정답」으로 갑니다.
3. 도전 문제는 안 풀고 넘어가도 됩니다.


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">5교시 (14:00–14:50) · 탐지 룰 ③</mark>


아래 두 셀을 먼저 실행합니다. 오전 파일을 여기서 다시 만듭니다. **오전을 못 끝냈어도 여기서부터 시작할 수 있습니다.**


In [ ]:
%%writefile raw_logs.txt
2026-09-29 09:02:11 INFO accepted login for kim.cs from 10.1.2.11
2026-09-29 09:12:00 WARN failed login for kim01 from 203.0.113.5
2026-09-29 09:15:31 INFO accepted login for lee.yh from 10.1.2.34
2026-09-29 09:16:02 INFO session closed for 10.1.2.34
2026-09-29 10:03:19 WARN failed login for park.js from 10.1.3.7
2026-09-29 10:03:31 INFO accepted login for park.js from 10.1.3.7
2026-09-29 11:20:55 INFO accepted login for choi.mk from 10.1.4.2
2026-09-29 12:40:12 INFO session closed for 10.1.4.2
2026-09-29 03:11:05 WARN failed login for admin from 211.45.12.9
2026-09-29 03:12:47 WARN failed login for admin from 211.45.12.9
2026-09-29 03:13:58 WARN failed login for admin from 211.45.12.9
2026-09-29 03:15:22 WARN failed login for admin from 211.45.12.9
2026-09-29 03:17:09 INFO accepted login for admin from 211.45.12.9
2026-09-29 14:05:38 INFO accepted login for jung.hw from 10.1.2.88
2026-09-29 22:14:03 WARN failed login for kim.cs from 185.220.101.34
2026-09-29 22:14:21 WARN failed login for lee.yh from 185.220.101.34
2026-09-29 22:14:40 WARN failed login for choi.mk from 185.220.101.34
2026-09-29 22:14:58 WARN failed login for jung.hw from 185.220.101.34
2026-09-29 22:15:12 INFO session closed for 185.220.101.34
2026-09-29 23:40:07 WARN failed login for invalid user guest from 185.220.101.34


In [ ]:
import re
import json

PATTERN = r"(?P<time>\d{2}:\d{2}:\d{2}) (?P<level>\w+) \w+ login for (?P<user>[\w.]+) from (?P<ip>[\d.]+)"

rows = []
unmatched = []

with open("raw_logs.txt", encoding="utf-8") as f:
    for line in f:
        m = re.search(PATTERN, line)
        if m:
            rows.append(m.groupdict())
        else:
            unmatched.append(line.strip())

with open("normalized_logs.json", "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)

print(f"정규화 {len(rows)}건 / 안 맞음 {len(unmatched)}건")


---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">4 · 실패가 아니라 성공을 의심한다</mark>


### 왜 필요한가

1. 오전의 룰 두 개는 모두 **실패**를 셌습니다. 실패가 잦으면 수상하다는 기준이었습니다.
2. 그런데 이 로그에는 실패 네 번 뒤에 **성공**이 한 번 있습니다. 새벽 3시 17분, 계정은 `admin` 입니다.
3. 실패는 막힌 것이고 **성공은 뚫린 것**입니다. 우리 룰은 뚫린 쪽을 못 보고 있었습니다.


### 이 시간에 나오는 말

| 말 | 뜻 |
|---|---|
| 룰 ③ 심야 접속 | 사람이 자는 시간대의 **로그인 성공**을 의심한다 |
| 업무 시간 | 정상이라고 보는 시간대. 몇 시부터 몇 시까지인지는 사람이 정한다 |


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.1 룰 ③ — 시간대로 거른다</mark>

시각은 `09:02:11` 처럼 글자로 들어 있습니다. 앞 두 글자가 **시**입니다.

```python
time = "03:17:09"
hour = int(time[:2])
print(hour)          # 3
```

`time[:2]` 는 「앞에서 두 글자」라는 뜻입니다. 콜론으로 나눠도 되고, 이렇게 잘라도 됩니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
time = "22:14:03"
print(int(time[:2]))
```

막히면 바로 위 `4.1 룰 ③ — 시간대로 거른다` 설명을 다시 봅니다.


In [ ]:
time = "22:14:03"
print(int(time[:2]))


✅ `22`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-2 · 무엇이 보일까요</font></h3></td></tr></table>

이번에는 읽어 온 기록에서 성공만 세어 봅니다.

```python
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

ok = 0
for row in rows:
    if row["level"] == "INFO":
        ok = ok + 1

print(ok)
```

막히면 바로 위 `4.1 룰 ③ — 시간대로 거른다` 설명을 다시 봅니다.


In [ ]:
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

ok = 0
for row in rows:
    if row["level"] == "INFO":
        ok = ok + 1

print(ok)


✅ `6`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-3 · 시만 꺼내 보기</font></h3></td></tr></table>

`rows` 를 돌면서 각 기록의 **시**를 숫자로 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `9` · `9` · `9` … 숫자 16줄 |

**💡 힌트**

1. `row["time"]` 이 `09:02:11` 같은 글자입니다.
2. 앞 두 글자는 `[:2]` 로 자릅니다.
3. `int()` 로 숫자로 바꿔 출력합니다.


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-4 · 새벽 기록만 고르기</font></h3></td></tr></table>

**06시 이전**의 기록만 골라 시각과 계정을 출력하시오. 성공·실패를 아직 가리지 않습니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `03:11:05 admin` 부터 `03:17:09 admin` 까지 다섯 줄 |

**💡 힌트**

1. 문제 4-3의 반복 안에 `if` 를 하나 넣습니다.
2. 06시 이전은 `hour < 6` 입니다.
3. `print(a, b)` 로 두 값을 한 줄에 냅니다.


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-5 · 룰 ③ 완성 — 새벽에 성공한 것</font></h3></td></tr></table>

새벽 기록 중 **성공(`INFO`)** 인 것만 골라 경보를 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[룰 3] 심야 접속: 03:17:09 admin (211.45.12.9)` |

**💡 힌트**

1. 문제 4-4에 조건을 하나 더합니다.
2. 두 조건은 `and` 로 잇습니다.
3. f-string 으로 문장을 만들어 출력합니다.


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-6 · 기준 시각을 맨 위로</font></h3></td></tr></table>

문제 4-5의 `6` 을 **`NIGHT_END`** 라는 이름으로 맨 위에 올리시오. 기준을 바꿀 때 한 곳만 고치면 되게 합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 문제 4-5와 같다 |

**💡 힌트**

1. 맨 위에 `NIGHT_END = 6` 을 적습니다.
2. `if` 줄의 숫자 자리를 그 이름으로 바꿉니다.
3. 오전 룰 ①의 `THRESHOLD` 와 같은 방식입니다.


In [ ]:
# 여기에 코드를 입력하세요


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다 — 8교시에 모아 푸는 시간이 있습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 4-1 · 기준을 바꿔 보기</font></h3></td></tr></table>

`NIGHT_END` 를 `6`·`9`·`12` 로 바꿔 각각 경보가 몇 건이 되는지 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 기준이 커질수록 경보가 는다 |

**💡 힌트**

1. 기준 세 개를 리스트에 담고 `for` 로 돕니다.
2. 경보 건수를 세는 숫자를 기준마다 새로 0으로 둡니다.
3. 기준을 넓히면 정상 접속까지 잡힙니다 — 헛경보입니다.


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.2 세 룰을 한 번에</mark>

룰 세 개는 **같은 `rows` 를 세 가지 각도로 보는 것**입니다. 한 번 읽어 두고 셋을 차례로 돌립니다.

| 룰 | 무엇을 본다 | 기준 |
|---|---|---|
| ① 브루트포스 | 계정별 실패 횟수 | 3회 이상 |
| ② 의심 IP | IP 마다 시도한 계정 수 | 2개 이상 |
| ③ 심야 접속 | 새벽의 로그인 성공 | 06시 이전 |

오전에 만든 룰 ①②의 코드를 그대로 가져와 ③ 아래에 붙이면 됩니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-7 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요.

```python
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

count = {}
for row in rows:
    if row["level"] == "WARN":
        user = row["user"]
        if user in count:
            count[user] = count[user] + 1
        else:
            count[user] = 1

print(count["admin"])
```

막히면 바로 위 `4.2 세 룰을 한 번에` 설명을 다시 봅니다.


In [ ]:
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

count = {}
for row in rows:
    if row["level"] == "WARN":
        user = row["user"]
        if user in count:
            count[user] = count[user] + 1
        else:
            count[user] = 1

print(count["admin"])


✅ `4`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-8 · 무엇이 보일까요</font></h3></td></tr></table>

이번에는 IP 쪽입니다.

```python
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

table = {}
for row in rows:
    ip = row["ip"]
    if ip in table:
        if row["user"] not in table[ip]:
            table[ip].append(row["user"])
    else:
        table[ip] = [row["user"]]

print(len(table["185.220.101.34"]))
```

막히면 바로 위 `4.2 세 룰을 한 번에` 설명을 다시 봅니다.


In [ ]:
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

table = {}
for row in rows:
    ip = row["ip"]
    if ip in table:
        if row["user"] not in table[ip]:
            table[ip].append(row["user"])
    else:
        table[ip] = [row["user"]]

print(len(table["185.220.101.34"]))


✅ `4`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-9 · 룰 ①② 를 다시 띄우기</font></h3></td></tr></table>

오전에 만든 룰 ①과 ②를 여기서 다시 돌려 경보 두 줄을 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[룰 1] 확인 필요: admin — 실패 4회` · `[룰 2] 의심 IP: 185.220.101.34 — 계정 4개 시도` |

**💡 힌트**

1. 오전 도전 ⭐3-2의 코드를 그대로 가져와도 됩니다.
2. 계정 세기와 IP 모으기를 한 반복 안에서 함께 합니다.
3. 출력은 반복이 끝난 뒤에 합니다.


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-10 · 세 줄을 한 번에</font></h3></td></tr></table>

룰 ①②③ 을 이어 붙여 경보 **세 줄**을 한 번에 출력하시오. 오늘 오후의 첫 도착점입니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[룰 1]` · `[룰 2]` · `[룰 3]` 세 줄 |

**💡 힌트**

1. 문제 4-9 뒤에 문제 4-6을 이어 붙이면 됩니다.
2. 파일을 읽는 줄은 맨 위에 한 번만 둡니다.
3. 기준값 두 개(`THRESHOLD`·`NIGHT_END`)를 맨 위에 나란히 둡니다.


In [ ]:
# 여기에 코드를 입력하세요


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 4-11 · 요약 한 줄 붙이기</font></h3></td></tr></table>

경보 세 줄 **위에** 요약 한 줄을 붙이시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[요약] 정규화 16건 / 경보 3건` 다음에 세 줄 |

**💡 힌트**

1. 경보를 바로 출력하지 말고 리스트에 모읍니다.
2. 모은 뒤 개수를 세어 요약을 먼저 출력합니다.
3. 그다음 리스트를 돌며 한 줄씩 출력합니다.


In [ ]:
# 여기에 코드를 입력하세요


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다 — 8교시에 모아 푸는 시간이 있습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 4-2 · 어느 룰이 몇 건인가</font></h3></td></tr></table>

룰마다 경보가 몇 건인지 따로 세어 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `룰 1 — 1건` · `룰 2 — 1건` · `룰 3 — 1건` |

**💡 힌트**

1. 룰마다 리스트를 따로 둡니다.
2. 또는 한 리스트에 담되 `if "[룰 1]" in line:` 으로 셉니다.
3. 세 줄을 차례로 출력합니다.


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.정리 📋 한눈에</mark>

| 쓰는 법 | 뜻 |
|---|---|
| `int(time[:2])` | 글자 시각에서 **시**만 숫자로 꺼낸다 |
| `hour < NIGHT_END` | 새벽인지 본다 |
| `level == "INFO"` | 성공한 접속인지 본다 |
| 기준값을 맨 위에 | 바꿀 때 한 곳만 고친다 |

**경보는 파일로 남기지 않습니다.** 내일 필요하면 `normalized_logs.json` 을 다시 읽어 룰을 한 번 더 돌립니다.


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">6교시 (15:00–15:50) · API 와 HTTP</mark>


---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">5 · 우리 파일 밖으로 나간다</mark>


### 왜 필요한가

1. 경보 세 줄을 띄웠습니다. 그런데 `185.220.101.34` 가 **정말 위험한 곳인지**는 우리 파일 어디에도 없습니다.
2. 바깥의 평판 데이터베이스에 물어봐야 합니다. 남의 컴퓨터에 들어가 볼 수는 없습니다.
3. 그래서 물어볼 수 있는 것을 미리 정해 **창구**로 열어 둡니다. 식당의 메뉴판과 같습니다.


### 이 시간에 나오는 말

| 말 | 뜻 |
|---|---|
| API | 남의 프로그램에 무엇을 물어볼지 정해 둔 창구 |
| HTTP | 그 창구에서 오가는 말의 규칙 |
| 요청 | 이쪽이 보내는 말 |
| 응답 | 저쪽이 돌려주는 말 |
| 메서드 | 무엇을 하려는지 밝히는 표시 (`GET`·`POST`) |


> **오늘은 실제로 요청을 보내지 않습니다.** 보내는 도구(`requests`)는 내일 배웁니다.
> 오늘은 **서버가 줬다고 치는 값**을 가지고 읽고 판단하는 연습만 합니다.


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">5.1 응답은 글자로 온다</mark>

서버가 돌려주는 것은 **글자 한 덩이**입니다. 어제 배운 `json.loads` 로 값으로 되돌립니다.

```python
import json

text = '{"city": "seoul", "count": 3}'
data = json.loads(text)

print(data["city"])        # seoul
print(data["count"] + 1)   # 4   ← 숫자는 숫자로 돌아온다
```

되돌린 결과는 **딕셔너리**입니다. 그래서 이름으로 바로 꺼낼 수 있습니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
import json

response_text = '{"ip": "185.220.101.34", "country": "NL", "risk": "high", "reports": 42}'
data = json.loads(response_text)

print(data["risk"])
```

막히면 바로 위 `5.1 응답은 글자로 온다` 설명을 다시 봅니다.


In [ ]:
import json

response_text = '{"ip": "185.220.101.34", "country": "NL", "risk": "high", "reports": 42}'
data = json.loads(response_text)

print(data["risk"])


✅ `high`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-2 · 무엇이 보일까요</font></h3></td></tr></table>

이번에는 숫자 칸을 꺼내 1을 더합니다.

```python
import json

response_text = '{"ip": "185.220.101.34", "country": "NL", "risk": "high", "reports": 42}'
data = json.loads(response_text)

print(data["reports"] + 1)
```

막히면 바로 위 `5.1 응답은 글자로 온다` 설명을 다시 봅니다.


In [ ]:
import json

response_text = '{"ip": "185.220.101.34", "country": "NL", "risk": "high", "reports": 42}'
data = json.loads(response_text)

print(data["reports"] + 1)


✅ `43`


글자 안에 있던 `42` 가 **숫자로** 돌아왔습니다. 그래서 더하기가 됩니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-3 · 나라 코드 꺼내기</font></h3></td></tr></table>

응답에서 **나라 코드**를 꺼내 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `NL` |

**💡 힌트**

1. `json.loads` 로 먼저 값으로 되돌립니다.
2. 되돌린 것은 딕셔너리입니다.
3. 칸 이름은 `country` 입니다.


In [ ]:
import json

response_text = '{"ip": "185.220.101.34", "country": "NL", "risk": "high", "reports": 42}'


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-4 · 위험하면 경보</font></h3></td></tr></table>

`risk` 가 `high` 이면 경보 한 줄을 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[조회] 185.220.101.34 — 위험 등급 high` |

**💡 힌트**

1. 되돌린 뒤 `if` 로 견줍니다.
2. 같은지 볼 때는 `==` 입니다.
3. f-string 으로 IP 와 등급을 함께 냅니다.


In [ ]:
import json

response_text = '{"ip": "185.220.101.34", "country": "NL", "risk": "high", "reports": 42}'


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-5 · 신고 건수로 판단하기</font></h3></td></tr></table>

`reports` 가 **10 이상**이면 `신고 많음`, 아니면 `신고 적음` 을 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `신고 많음` |

**💡 힌트**

1. 숫자 비교는 `>=` 입니다.
2. `if` 아래에 `else:` 를 같은 들여쓰기로 둡니다.
3. 되돌린 값이 숫자라서 따옴표 없이 견줍니다.


In [ ]:
import json

response_text = '{"ip": "185.220.101.34", "country": "NL", "risk": "high", "reports": 42}'


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다 — 8교시에 모아 푸는 시간이 있습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 5-1 · 응답 세 개를 한 번에</font></h3></td></tr></table>

응답 글자 세 개를 리스트에 담고, `risk` 가 `high` 인 것만 골라 IP 를 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `185.220.101.34` · `203.0.113.5` |

**💡 힌트**

1. 리스트에는 글자 그대로 담습니다.
2. `for` 안에서 하나씩 `json.loads` 합니다.
3. 되돌린 뒤 `if` 로 거릅니다.


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">5.2 주소를 쪼개 읽는다 · 메서드</mark>

요청을 보낼 주소는 네 조각으로 되어 있습니다.

```
https://api.example.com/v1/ip/185.220.101.34?full=true
└방식┘ └───호스트────┘└─────경로─────┘└─질의─┘
```

| 조각 | 무엇 |
|---|---|
| 방식 | `https` — 어떤 규칙으로 말할지 |
| 호스트 | `api.example.com` — 어느 서버에 |
| 경로 | `/v1/ip/185.220.101.34` — 그 서버의 어느 창구에 |
| 질의 | `full=true` — 창구에 덧붙이는 조건 |

**메서드**는 무엇을 하려는지 밝히는 표시입니다.

| 메서드 | 무엇 | 두 번 보내면 |
|---|---|---|
| `GET` | 조회한다 | 결과가 같다 |
| `POST` | 새로 만든다 | 두 개가 생긴다 |

오늘 우리가 하려는 일은 **조회**라서 `GET` 입니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-6 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요.

```python
url = "https://api.example.com/v1/ip/185.220.101.34?full=true"

print(url.split("?")[0])
```

막히면 바로 위 `5.2 주소를 쪼개 읽는다` 설명을 다시 봅니다.


In [ ]:
url = "https://api.example.com/v1/ip/185.220.101.34?full=true"

print(url.split("?")[0])


✅ `https://api.example.com/v1/ip/185.220.101.34`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-7 · 무엇이 보일까요</font></h3></td></tr></table>

이번에는 슬래시로 쪼갭니다.

```python
url = "https://api.example.com/v1/ip/185.220.101.34?full=true"

print(url.split("/")[2])
```

막히면 바로 위 `5.2 주소를 쪼개 읽는다` 설명을 다시 봅니다.


In [ ]:
url = "https://api.example.com/v1/ip/185.220.101.34?full=true"

print(url.split("/")[2])


✅ `api.example.com`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-8 · 질의만 꺼내기</font></h3></td></tr></table>

주소에서 **질의 부분**만 꺼내 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `full=true` |

**💡 힌트**

1. 물음표로 쪼개면 두 조각이 됩니다.
2. 앞 조각이 주소, 뒤 조각이 질의입니다.
3. 뒤 조각의 번호는 1입니다.


In [ ]:
url = "https://api.example.com/v1/ip/185.220.101.34?full=true"


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-9 · 경로 맨 뒤의 IP 꺼내기</font></h3></td></tr></table>

주소의 **경로 맨 뒤**에 있는 IP 를 꺼내 출력하시오.

- 질의를 먼저 떼어 낸 뒤 슬래시로 쪼갭니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `185.220.101.34` |

**💡 힌트**

1. 먼저 `?` 로 쪼개 앞 조각만 남깁니다.
2. 그 조각을 `/` 로 다시 쪼갭니다.
3. 맨 뒤 번호는 `len(...) - 1` 입니다.


In [ ]:
url = "https://api.example.com/v1/ip/185.220.101.34?full=true"


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-10 · 조회 주소 만들기</font></h3></td></tr></table>

IP 하나를 받아 **조회 주소**를 만들어 출력하시오.

| | |
|---|---|
| 주어지는 값 | `ip = "211.45.12.9"` |
| 🎯 나와야 하는 결과 | `https://api.example.com/v1/ip/211.45.12.9?full=true` |

**💡 힌트**

1. f-string 안에 중괄호로 IP 를 넣습니다.
2. 앞뒤 글자는 그대로 적습니다.
3. 주소를 이름 하나에 담아 두고 출력합니다.


In [ ]:
ip = "211.45.12.9"


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-11 · 경보에 나온 IP 로 주소 만들기</font></h3></td></tr></table>

오전 룰 ②가 찾은 IP 목록을 돌면서 **조회 주소를 한 줄씩** 출력하시오.

| | |
|---|---|
| 주어지는 값 | `ips = ["185.220.101.34", "211.45.12.9"]` |
| 🎯 나와야 하는 결과 | 주소 두 줄 |

**💡 힌트**

1. `for` 로 목록을 하나씩 돕니다.
2. 문제 5-10의 f-string 을 반복 안에 둡니다.
3. 출력도 반복 안에 둡니다.


In [ ]:
ips = ["185.220.101.34", "211.45.12.9"]


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다 — 8교시에 모아 푸는 시간이 있습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 5-2 · 조회 목록을 파일로</font></h3></td></tr></table>

문제 5-11에서 만든 주소들을 `lookup_urls.txt` 로 저장하시오. 내일 이 목록을 실제로 보냅니다.

| | |
|---|---|
| 🎯 확인 | `!cat lookup_urls.txt` 로 두 줄이 보인다 |

**💡 힌트**

1. 파일 쓰기는 어제 배운 `open(이름, w, encoding=utf-8)` 입니다.
2. 한 줄씩 쓸 때는 끝에 줄바꿈을 붙입니다.
3. `chr(10)` 이 줄바꿈 한 글자입니다.


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">5.정리 📋 한눈에</mark>

| 쓰는 법 | 뜻 |
|---|---|
| `json.loads(글자)` | 응답 글자를 딕셔너리로 되돌린다 |
| `url.split("?")` | 주소와 질의를 가른다 |
| `url.split("/")[2]` | 호스트를 꺼낸다 |
| `GET` / `POST` | 조회한다 / 새로 만든다 |


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">7교시 (16:00–16:50) · 상태코드와 인증</mark>


---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">6 · 답이 안 올 때를 읽는다</mark>


### 왜 필요한가

1. 요청을 보냈는데 원하는 값이 안 왔습니다. 서버가 죽은 것인지, 주소가 틀린 것인지, 권한이 없는 것인지 알아야 **다음 행동이 정해집니다.**
2. 그래서 응답에는 세 자리 숫자가 함께 옵니다. **앞자리 하나**만 봐도 어느 쪽 잘못인지 갈립니다.


### 이 시간에 나오는 말

| 말 | 뜻 |
|---|---|
| 상태코드 | 요청이 어떻게 됐는지 알리는 세 자리 숫자 |
| 인증 | 누구인지 밝히는 일 |
| 인가 | 그 사람에게 권한이 있는지 보는 일 |
| API 키 | 발급받은 긴 글자를 요청마다 붙이는 방식 |
| 토큰 | 먼저 받아 두고 붙이는, 유효 기간이 있는 글자 |


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">6.1 상태코드 — 앞자리로 좁힌다</mark>

| 앞자리 | 뜻 | 예 |
|---|---|---|
| `2` | 성공 | `200` 처리했다 · `201` 새로 만들었다 |
| `4` | **보낸 쪽** 잘못 | `401` `403` `404` |
| `5` | **서버** 잘못 | `500` 서버가 터졌다 |

앞자리는 글자로 바꿔 첫 글자를 보면 됩니다.

```python
code = 404
print(str(code)[0])     # 4
```

`str()` 은 숫자를 글자로 바꾸는 명령입니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요.

```python
code = 503

print(str(code)[0])
```

막히면 바로 위 `6.1 상태코드 — 앞자리로 좁힌다` 설명을 다시 봅니다.


In [ ]:
code = 503

print(str(code)[0])


✅ `5`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-2 · 무엇이 보일까요</font></h3></td></tr></table>

이번에는 목록을 돌면서 앞자리만 봅니다.

```python
codes = [200, 404, 500]

for code in codes:
    print(str(code)[0])
```

막히면 바로 위 `6.1 상태코드 — 앞자리로 좁힌다` 설명을 다시 봅니다.


In [ ]:
codes = [200, 404, 500]

for code in codes:
    print(str(code)[0])


✅ `2 · 4 · 5 세 줄`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-3 · 성공인지 보기</font></h3></td></tr></table>

상태코드 하나를 받아 앞자리가 `2` 이면 `성공` 을 출력하시오.

| | |
|---|---|
| 주어지는 값 | `code = 201` |
| 🎯 나와야 하는 결과 | `성공` |

**💡 힌트**

1. `str(code)[0]` 이 앞자리 한 글자입니다.
2. 앞자리는 **글자**라서 `"2"` 처럼 따옴표를 붙여 견줍니다.
3. `if` 줄 끝에 콜론을 붙입니다.


In [ ]:
code = 201


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-4 · 세 갈래로 나누기</font></h3></td></tr></table>

상태코드를 받아 앞자리에 따라 셋으로 갈라 출력하시오.

| 앞자리 | 출력할 말 |
|---|---|
| `2` | `성공` |
| `4` | `내가 보낸 요청이 잘못됐다` |
| `5` | `서버 쪽 문제다` |

| | |
|---|---|
| 주어지는 값 | `code = 403` |
| 🎯 나와야 하는 결과 | `내가 보낸 요청이 잘못됐다` |

**💡 힌트**

1. `if` → `elif` → `else` 로 세 갈래를 만듭니다.
2. 견주는 값은 모두 글자입니다.
3. 처음 참이 된 갈래 하나만 실행됩니다.


In [ ]:
code = 403


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-5 · 실패한 요청 세기</font></h3></td></tr></table>

응답 목록에서 **앞자리가 4인 것**이 몇 개인지 세어 출력하시오.

| | |
|---|---|
| 주어지는 값 | `codes = [200, 404, 500, 403, 200, 401]` |
| 🎯 나와야 하는 결과 | `보낸 쪽 잘못 3건` |

**💡 힌트**

1. 숫자를 담을 이름을 반복 전에 만듭니다.
2. 반복 안에서 앞자리를 견줍니다.
3. 출력은 반복이 끝난 뒤 한 번만 합니다.


In [ ]:
codes = [200, 404, 500, 403, 200, 401]


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-6 · 코드마다 뜻 붙이기</font></h3></td></tr></table>

상태코드와 뜻을 담은 딕셔너리를 만들고, 목록을 돌면서 **코드와 뜻**을 한 줄씩 출력하시오.

- 목록에 없는 코드는 `알 수 없음` 으로 냅니다.

| | |
|---|---|
| 주어지는 값 | `codes = [200, 404, 999]` |
| 🎯 나와야 하는 결과 | `200 처리했다` · `404 그런 주소가 없다` · `999 알 수 없음` |

**💡 힌트**

1. 뜻을 담은 딕셔너리를 먼저 만듭니다.
2. 있는 코드인지는 `if code in meaning:` 으로 확인합니다.
3. 없으면 `else` 로 갑니다.


In [ ]:
codes = [200, 404, 999]


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다 — 8교시에 모아 푸는 시간이 있습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 6-1 · 앞자리별로 세기</font></h3></td></tr></table>

응답 목록을 앞자리별로 세어 출력하시오.

| | |
|---|---|
| 주어지는 값 | `codes = [200, 404, 500, 403, 200, 401]` |
| 🎯 나와야 하는 결과 | `2 — 2건` · `4 — 3건` · `5 — 1건` |

**💡 힌트**

1. 빈 딕셔너리를 반복 전에 만듭니다.
2. 키는 앞자리 글자입니다.
3. 이미 있는 키인지 `if first in count:` 로 확인합니다.


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">6.2 401 ↔ 403 · 인증 두 방식</mark>

둘 다 막히는 상황이지만 **원인이 다릅니다.**

| 견주는 점 | `401` | `403` |
|---|---|---|
| 서버가 하는 말 | 당신이 누구인지 **모르겠다** | 누구인지는 알겠는데 **안 된다** |
| 어디서 막혔나 | 인증 | 인가 |
| 할 일 | 토큰을 다시 받아 붙인다 | 담당자에게 권한을 요청한다 |

누구인지 밝히는 방식은 크게 둘입니다.

| 방식 | 어떻게 | 약점 |
|---|---|---|
| API 키 | 발급받은 긴 글자를 요청마다 붙인다 | 한 번 새면 바꾸기 전까지 계속 쓰인다 |
| 토큰 | 먼저 받아 두고 붙인다 | 유효 기간이 있어 피해가 시간에 갇힌다 |

붙이는 자리는 **헤더**입니다. `Authorization: Bearer 토큰` 꼴로 적습니다.

> **키를 코드 밖에 두는 방법(환경변수)은 내일 배웁니다.** 실제로 키를 쓰는 것이 내일이기 때문입니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-7 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요.

```python
todo = {401: "토큰을 다시 받는다", 403: "권한을 요청한다"}

print(todo[401])
```

막히면 바로 위 `6.2 401 ↔ 403 · 인증 두 방식` 설명을 다시 봅니다.


In [ ]:
todo = {401: "토큰을 다시 받는다", 403: "권한을 요청한다"}

print(todo[401])


✅ `토큰을 다시 받는다`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-8 · 무엇이 보일까요</font></h3></td></tr></table>

이번에는 헤더를 만들어 봅니다.

```python
token = "abc123"
headers = {"Authorization": "Bearer " + token}

print(headers)
```

막히면 바로 위 `6.2 401 ↔ 403 · 인증 두 방식` 설명을 다시 봅니다.


In [ ]:
token = "abc123"
headers = {"Authorization": "Bearer " + token}

print(headers)


✅ `{'Authorization': 'Bearer abc123'}`


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-9 · 막힌 이유에 따라 할 일 내기</font></h3></td></tr></table>

상태코드를 받아 `401` 이면 `토큰을 다시 받는다`, `403` 이면 `권한을 요청한다` 를 출력하시오.

| | |
|---|---|
| 주어지는 값 | `code = 403` |
| 🎯 나와야 하는 결과 | `권한을 요청한다` |

**💡 힌트**

1. `if` 와 `elif` 로 두 갈래를 만듭니다.
2. 상태코드는 숫자라서 따옴표 없이 견줍니다.
3. 둘 다 아니면 `else` 로 `다른 문제다` 를 냅니다.


In [ ]:
code = 403


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-10 · 목록에 할 일 붙이기</font></h3></td></tr></table>

응답 목록을 돌면서 코드마다 **할 일**을 한 줄씩 출력하시오.

| | |
|---|---|
| 주어지는 값 | `codes = [200, 401, 403, 500]` |
| 🎯 나와야 하는 결과 | `200 그대로 쓴다` · `401 토큰을 다시 받는다` · `403 권한을 요청한다` · `500 잠시 뒤 다시 보낸다` |

**💡 힌트**

1. 할 일을 담은 딕셔너리를 먼저 만듭니다.
2. 키는 상태코드 숫자입니다.
3. 있는 코드인지 `if code in todo:` 로 확인합니다.


In [ ]:
codes = [200, 401, 403, 500]


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-11 · 요청에 붙일 헤더 만들기</font></h3></td></tr></table>

토큰을 받아 요청에 붙일 **헤더 딕셔너리**를 만들어 출력하시오.

| | |
|---|---|
| 주어지는 값 | `token = "sk-0929-demo"` |
| 🎯 나와야 하는 결과 | `{'Authorization': 'Bearer sk-0929-demo'}` |

**💡 힌트**

1. 칸 이름은 `Authorization` 입니다.
2. 값은 `Bearer ` 뒤에 토큰을 이어 붙인 글자입니다.
3. 글자끼리는 `+` 로 이어 붙입니다. f-string 으로 해도 됩니다.


In [ ]:
token = "sk-0929-demo"


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다 — 8교시에 모아 푸는 시간이 있습니다.</div>


<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 6-2 · 조회 결과를 경보에 붙이기</font></h3></td></tr></table>

상태코드와 응답을 함께 받아, **성공일 때만** 위험 등급을 경보에 붙여 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[조회] 185.220.101.34 — 위험 등급 high` 와 `[조회 실패] 211.45.12.9 — 코드 403` |

**💡 힌트**

1. 응답 두 개를 리스트에 담고 `for` 로 돕니다.
2. 앞자리가 `2` 일 때만 `json.loads` 를 부릅니다.
3. 실패한 것은 코드만 내면 됩니다.


In [ ]:
# 여기에 코드를 입력하세요


### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">6.정리 📋 한눈에</mark>

| 쓰는 법 | 뜻 |
|---|---|
| `str(code)[0]` | 상태코드 앞자리 |
| `2` / `4` / `5` | 성공 / 보낸 쪽 잘못 / 서버 잘못 |
| `401` | 누구인지 모르겠다 → 토큰을 다시 |
| `403` | 누구인지는 알겠는데 안 된다 → 권한 요청 |
| `{"Authorization": "Bearer " + token}` | 요청에 붙이는 헤더 |


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">8교시 (17:00–17:30) · 마무리</mark>


새로 배우는 것이 없습니다. 오늘 만든 것을 확인하고 마무리합니다.

### 8.1 오늘 만든 것

| 파일 | 무엇 | 언제 |
|---|---|---|
| `raw_logs.txt` | 쉼표가 없는 서버 로그 원본 | 오전 2교시 |
| `normalize_logs.py` | 정규식으로 한 모양으로 맞추는 프로그램 | 오전 3교시 |
| `normalized_logs.json` | 정규화한 기록 16건 | 오전 3교시 |
| `unmatched_logs.txt` | 규칙에 안 맞은 줄 4건 | 오전 3교시 |

드라이브 `agent_core` 폴더를 열어 네 개가 다 있는지 봅니다. 없으면 오전 문제 3-10 으로 돌아가 다시 합니다.


### 8.2 경보는 왜 파일로 안 남겼나

오늘 만든 탐지 룰 세 개는 화면에만 찍었습니다. 학원 교안이 정한 오늘 산출물은 위의 네 개뿐이고, **교안에 없는 이름을 우리가 새로 짓지 않기** 때문입니다.

내일 조회할 IP 가 필요해지면 **`normalized_logs.json` 을 다시 읽어 룰 ②를 한 번 더 돌리면 됩니다.** 오늘 문제 4-9 가 그 코드입니다.


### 8.3 남은 ⭐도전 문제

넘어간 도전 문제를 여기서 풉니다. **다 풀지 않아도 됩니다.** 기본 문제를 못 끝낸 분은 도전이 아니라 기본부터 합니다.

| 어디 | 도전 문제 |
|---|---|
| 오전 2교시 | ⭐1-1 계정이 아닌 줄 세기 · ⭐1-2 WARN 인 줄 세기 |
| 오전 3교시 | ⭐2-1 번호 바꿔 보기 · ⭐2-2 계정만 모아 보기 · ⭐2-3 안 맞는 줄 읽기 |
| 오전 4교시 | ⭐3-1 임계값 바꿔 보기 · ⭐3-2 두 룰을 한 번에 |
| 오후 5교시 | ⭐4-1 기준을 바꿔 보기 · ⭐4-2 어느 룰이 몇 건인가 |
| 오후 6교시 | ⭐5-1 응답 세 개를 한 번에 · ⭐5-2 조회 목록을 파일로 |
| 오후 7교시 | ⭐6-1 앞자리별로 세기 · ⭐6-2 조회 결과를 경보에 붙이기 |


### 8.4 오늘 배운 것 한눈에

| 시간 | 배운 것 |
|---|---|
| 오전 | `split` 이 조용히 틀린다 · `re.search` 와 기호 · 그룹과 `.groupdict()` · 안 맞는 줄 남기기 · 탐지 룰 ①② |
| 오후 | 탐지 룰 ③ 심야 성공 · 요청과 응답 · 주소 쪼개 읽기 · 상태코드 · `401` ↔ `403` |

**내일(9/30) 오전**에는 오늘 흉내만 낸 요청을 `requests` 로 실제로 보냅니다. API 키를 코드 밖에 두는 법도 그때 배웁니다.


---

# <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">정답 · 먼저 풀어 본 뒤에 엽니다</mark>

각 셀은 제목만 보입니다. 「코드 표시」를 누르면 코드가 열립니다.


In [ ]:
#@title 정답 4-3 { display-mode: "form" }
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

for row in rows:
    print(int(row["time"][:2]))


In [ ]:
#@title 정답 4-4 { display-mode: "form" }
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

for row in rows:
    hour = int(row["time"][:2])
    if hour < 6:
        print(row["time"], row["user"])


In [ ]:
#@title 정답 4-5 { display-mode: "form" }
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

for row in rows:
    hour = int(row["time"][:2])
    if hour < 6 and row["level"] == "INFO":
        print(f"[룰 3] 심야 접속: {row['time']} {row['user']} ({row['ip']})")


In [ ]:
#@title 정답 4-6 { display-mode: "form" }
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

NIGHT_END = 6

for row in rows:
    hour = int(row["time"][:2])
    if hour < NIGHT_END and row["level"] == "INFO":
        print(f"[룰 3] 심야 접속: {row['time']} {row['user']} ({row['ip']})")


In [ ]:
#@title 정답 ⭐4-1 { display-mode: "form" }
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

for night_end in [6, 9, 12]:
    hit = 0
    for row in rows:
        hour = int(row["time"][:2])
        if hour < night_end and row["level"] == "INFO":
            hit = hit + 1
    print(f"기준 {night_end}시 — 경보 {hit}건")


In [ ]:
#@title 정답 4-9 { display-mode: "form" }
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

THRESHOLD = 3

count = {}
table = {}

for row in rows:
    user = row["user"]
    ip = row["ip"]
    if row["level"] == "WARN":
        if user in count:
            count[user] = count[user] + 1
        else:
            count[user] = 1
    if ip in table:
        if user not in table[ip]:
            table[ip].append(user)
    else:
        table[ip] = [user]

for user in count:
    if count[user] >= THRESHOLD:
        print(f"[룰 1] 확인 필요: {user} — 실패 {count[user]}회")

for ip in table:
    if len(table[ip]) >= 2:
        print(f"[룰 2] 의심 IP: {ip} — 계정 {len(table[ip])}개 시도")


In [ ]:
#@title 정답 4-10 { display-mode: "form" }
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

THRESHOLD = 3
NIGHT_END = 6

count = {}
table = {}

for row in rows:
    user = row["user"]
    ip = row["ip"]
    if row["level"] == "WARN":
        if user in count:
            count[user] = count[user] + 1
        else:
            count[user] = 1
    if ip in table:
        if user not in table[ip]:
            table[ip].append(user)
    else:
        table[ip] = [user]

for user in count:
    if count[user] >= THRESHOLD:
        print(f"[룰 1] 확인 필요: {user} — 실패 {count[user]}회")

for ip in table:
    if len(table[ip]) >= 2:
        print(f"[룰 2] 의심 IP: {ip} — 계정 {len(table[ip])}개 시도")

for row in rows:
    if int(row["time"][:2]) < NIGHT_END and row["level"] == "INFO":
        print(f"[룰 3] 심야 접속: {row['time']} {row['user']} ({row['ip']})")


In [ ]:
#@title 정답 4-11 { display-mode: "form" }
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

THRESHOLD = 3
NIGHT_END = 6

count = {}
table = {}

for row in rows:
    user = row["user"]
    ip = row["ip"]
    if row["level"] == "WARN":
        if user in count:
            count[user] = count[user] + 1
        else:
            count[user] = 1
    if ip in table:
        if user not in table[ip]:
            table[ip].append(user)
    else:
        table[ip] = [user]

alerts = []

for user in count:
    if count[user] >= THRESHOLD:
        alerts.append(f"[룰 1] 확인 필요: {user} — 실패 {count[user]}회")

for ip in table:
    if len(table[ip]) >= 2:
        alerts.append(f"[룰 2] 의심 IP: {ip} — 계정 {len(table[ip])}개 시도")

for row in rows:
    if int(row["time"][:2]) < NIGHT_END and row["level"] == "INFO":
        alerts.append(f"[룰 3] 심야 접속: {row['time']} {row['user']} ({row['ip']})")

print(f"[요약] 정규화 {len(rows)}건 / 경보 {len(alerts)}건")
for line in alerts:
    print(line)


In [ ]:
#@title 정답 ⭐4-2 { display-mode: "form" }
import json

with open("normalized_logs.json", encoding="utf-8") as f:
    rows = json.load(f)

THRESHOLD = 3
NIGHT_END = 6

count = {}
table = {}
for row in rows:
    user = row["user"]
    ip = row["ip"]
    if row["level"] == "WARN":
        if user in count:
            count[user] = count[user] + 1
        else:
            count[user] = 1
    if ip in table:
        if user not in table[ip]:
            table[ip].append(user)
    else:
        table[ip] = [user]

alerts = []
for user in count:
    if count[user] >= THRESHOLD:
        alerts.append("[룰 1] " + user)
for ip in table:
    if len(table[ip]) >= 2:
        alerts.append("[룰 2] " + ip)
for row in rows:
    if int(row["time"][:2]) < NIGHT_END and row["level"] == "INFO":
        alerts.append("[룰 3] " + row["user"])

for name in ["[룰 1]", "[룰 2]", "[룰 3]"]:
    hit = 0
    for line in alerts:
        if name in line:
            hit = hit + 1
    print(f"{name} — {hit}건")


In [ ]:
#@title 정답 5-3 { display-mode: "form" }
import json

response_text = '{"ip": "185.220.101.34", "country": "NL", "risk": "high", "reports": 42}'

data = json.loads(response_text)

print(data["country"])


In [ ]:
#@title 정답 5-4 { display-mode: "form" }
import json

response_text = '{"ip": "185.220.101.34", "country": "NL", "risk": "high", "reports": 42}'

data = json.loads(response_text)

if data["risk"] == "high":
    print(f"[조회] {data['ip']} — 위험 등급 {data['risk']}")


In [ ]:
#@title 정답 5-5 { display-mode: "form" }
import json

response_text = '{"ip": "185.220.101.34", "country": "NL", "risk": "high", "reports": 42}'

data = json.loads(response_text)

if data["reports"] >= 10:
    print("신고 많음")
else:
    print("신고 적음")


In [ ]:
#@title 정답 ⭐5-1 { display-mode: "form" }
import json

responses = [
    '{"ip": "185.220.101.34", "risk": "high"}',
    '{"ip": "10.1.2.11", "risk": "low"}',
    '{"ip": "203.0.113.5", "risk": "high"}',
]

for text in responses:
    data = json.loads(text)
    if data["risk"] == "high":
        print(data["ip"])


In [ ]:
#@title 정답 5-8 { display-mode: "form" }
url = "https://api.example.com/v1/ip/185.220.101.34?full=true"

print(url.split("?")[1])


In [ ]:
#@title 정답 5-9 { display-mode: "form" }
url = "https://api.example.com/v1/ip/185.220.101.34?full=true"

address = url.split("?")[0]
parts = address.split("/")

print(parts[len(parts) - 1])


In [ ]:
#@title 정답 5-10 { display-mode: "form" }
ip = "211.45.12.9"

url = f"https://api.example.com/v1/ip/{ip}?full=true"

print(url)


In [ ]:
#@title 정답 5-11 { display-mode: "form" }
ips = ["185.220.101.34", "211.45.12.9"]

for ip in ips:
    url = f"https://api.example.com/v1/ip/{ip}?full=true"
    print(url)


In [ ]:
#@title 정답 ⭐5-2 { display-mode: "form" }
ips = ["185.220.101.34", "211.45.12.9"]

with open("lookup_urls.txt", "w", encoding="utf-8") as f:
    for ip in ips:
        f.write(f"https://api.example.com/v1/ip/{ip}?full=true" + chr(10))

print("저장했습니다")


In [ ]:
#@title 정답 6-3 { display-mode: "form" }
code = 201

if str(code)[0] == "2":
    print("성공")


In [ ]:
#@title 정답 6-4 { display-mode: "form" }
code = 403

first = str(code)[0]

if first == "2":
    print("성공")
elif first == "4":
    print("내가 보낸 요청이 잘못됐다")
else:
    print("서버 쪽 문제다")


In [ ]:
#@title 정답 6-5 { display-mode: "form" }
codes = [200, 404, 500, 403, 200, 401]

bad = 0

for code in codes:
    if str(code)[0] == "4":
        bad = bad + 1

print(f"보낸 쪽 잘못 {bad}건")


In [ ]:
#@title 정답 6-6 { display-mode: "form" }
codes = [200, 404, 999]

meaning = {200: "처리했다", 404: "그런 주소가 없다", 500: "서버가 터졌다"}

for code in codes:
    if code in meaning:
        print(code, meaning[code])
    else:
        print(code, "알 수 없음")


In [ ]:
#@title 정답 ⭐6-1 { display-mode: "form" }
codes = [200, 404, 500, 403, 200, 401]

count = {}

for code in codes:
    first = str(code)[0]
    if first in count:
        count[first] = count[first] + 1
    else:
        count[first] = 1

for first in count:
    print(f"{first} — {count[first]}건")


In [ ]:
#@title 정답 6-9 { display-mode: "form" }
code = 403

if code == 401:
    print("토큰을 다시 받는다")
elif code == 403:
    print("권한을 요청한다")
else:
    print("다른 문제다")


In [ ]:
#@title 정답 6-10 { display-mode: "form" }
codes = [200, 401, 403, 500]

todo = {
    200: "그대로 쓴다",
    401: "토큰을 다시 받는다",
    403: "권한을 요청한다",
    500: "잠시 뒤 다시 보낸다",
}

for code in codes:
    if code in todo:
        print(code, todo[code])
    else:
        print(code, "알 수 없음")


In [ ]:
#@title 정답 6-11 { display-mode: "form" }
token = "sk-0929-demo"

headers = {"Authorization": "Bearer " + token}

print(headers)


In [ ]:
#@title 정답 ⭐6-2 { display-mode: "form" }
import json

results = [
    {"ip": "185.220.101.34", "code": 200, "body": '{"risk": "high"}'},
    {"ip": "211.45.12.9", "code": 403, "body": ""},
]

for result in results:
    if str(result["code"])[0] == "2":
        data = json.loads(result["body"])
        print(f"[조회] {result['ip']} — 위험 등급 {data['risk']}")
    else:
        print(f"[조회 실패] {result['ip']} — 코드 {result['code']}")
